# 连续词袋模型

## 第一阶段：项目准备与数据获取
首先，我们需要导入必要的库。

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import re
import requests
from collections import Counter

# 1. 配置参数 (Hyperparameters)
# 在工业级代码中，这些通常放在一个 config 文件或类中
class Config:
    EMBED_DIM = 100       # 词向量维度
    CONTEXT_SIZE = 2      # 上下文窗口大小 (左右各取2个词)
    BATCH_SIZE = 128      # 批次大小
    LEARNING_RATE = 0.001 # 学习率
    EPOCHS = 10           # 训练轮数
    MIN_FREQ = 2          # 过滤掉出现次数少于2次的低频词

config = Config()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 第二阶段：数据清洗与词表构建 (Preprocessing)
原始文本是无法直接喂给神经网络的。我们需要做三件事：
1. 清洗：去除标点、转小写。
2. 构建词表：给每个词分配唯一的 ID。
3. 生成训练对：(Context, Target)。

In [2]:
# 2. 获取并清洗数据
def get_alice_text():
    url = "https://www.gutenberg.org/files/11/11-0.txt"
    response = requests.get(url)
    text = response.text
    
    # 去除古腾堡项目的头尾声明，只保留正文（大约的切分）
    start = text.find("*** START OF THE PROJECT GUTENBERG EBOOK")
    end = text.find("*** END OF THE PROJECT GUTENBERG EBOOK")
    if start != -1 and end != -1:
        text = text[start:end]
    
    # 正则清洗：只保留字母，转小写
    text = text.lower()
    tokens = re.findall(r'\b[a-z]+\b', text)
    return tokens

print("Downloading and processing text...")
raw_tokens = get_alice_text()
print(f"Total tokens: {len(raw_tokens)}")

# 3. 构建词汇表 (Vocabulary)
# 工业级处理：使用 <UNK> 处理生僻词
vocab_counter = Counter(raw_tokens)
vocab = {"<UNK>": 0}
idx_to_word = {0: "<UNK>"}

for word, count in vocab_counter.items():
    if count >= config.MIN_FREQ:
        idx = len(vocab)
        vocab[word] = idx
        idx_to_word[idx] = word

VOCAB_SIZE = len(vocab)
print(f"Vocabulary size: {VOCAB_SIZE}")

# 将文本转换为索引列表
encoded_text = [vocab.get(token, vocab["<UNK>"]) for token in raw_tokens]

Total tokens: 27181
Vocabulary size: 1465


## 第三阶段：构建 PyTorch Dataset (Dataset Pipeline)
PyTorch 的核心优势在于 Dataset 和 DataLoader。这能帮我们自动处理 Batch（批次）和 Shuffle（打乱），这是工业级训练的基础。
CBOW 的数据格式是：
* 输入 (X): 上下文单词的索引列表 $[w_{t-2}, w_{t-1}, w_{t+1}, w_{t+2}]$
* 标签 (Y): 中心词的索引 $w_t$

In [3]:
class CBOWDataset(Dataset):
    def __init__(self, encoded_text, context_size):
        self.data = []
        # 滑动窗口生成数据
        for i in range(context_size, len(encoded_text) - context_size):
            target = encoded_text[i]
            # 获取上下文：左边 context_size 个 + 右边 context_size 个
            context = (
                encoded_text[i - context_size : i] + 
                encoded_text[i + 1 : i + context_size + 1]
            )
            self.data.append((context, target))
            
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        context, target = self.data[idx]
        return torch.tensor(context), torch.tensor(target)

# 实例化 Dataset 和 DataLoader
dataset = CBOWDataset(encoded_text, config.CONTEXT_SIZE)
dataloader = DataLoader(dataset, batch_size=config.BATCH_SIZE, shuffle=True)

print(f"Number of training pairs: {len(dataset)}")
# 检查一个样本
sample_context, sample_target = dataset[0]
print(f"Sample Context (indices): {sample_context}")
print(f"Sample Target (index): {sample_target}")

Number of training pairs: 27177
Sample Context (indices): tensor([0, 1, 0, 0])
Sample Target (index): 2


## 第四阶段：搭建模型 (Model Architecture)
这是最核心的部分。Embedding的实现见`词嵌入-跳元模型-demo`。

In [4]:
class CBOW(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super(CBOW, self).__init__()
        # 1. 词嵌入层：V -> N
        self.embeddings = nn.Embedding(vocab_size, embed_dim)
        
        # 2. 线性层：N -> V (将隐藏层映射回词表大小，用于预测)
        self.linear = nn.Linear(embed_dim, vocab_size)
        
    def forward(self, inputs):
        # inputs shape: [batch_size, context_window * 2]
        
        # 1. 查找嵌入
        # out shape: [batch_size, context_window * 2, embed_dim]
        embeds = self.embeddings(inputs)
        
        # 2. 聚合上下文 (CBOW的核心：求平均或求和)
        # out shape: [batch_size, embed_dim]
        # dim=1 表示在上下文单词数量这个维度上取平均
        hidden = torch.mean(embeds, dim=1)
        
        # 3. 预测层
        # out shape: [batch_size, vocab_size]
        output = self.linear(hidden)
        
        return output

model = CBOW(VOCAB_SIZE, config.EMBED_DIM).to(device)
print(model)

CBOW(
  (embeddings): Embedding(1465, 100)
  (linear): Linear(in_features=100, out_features=1465, bias=True)
)


## 第五阶段：训练循环 (Training Loop)
这里我们使用 CrossEntropyLoss。

注意：nn.CrossEntropyLoss 在内部已经包含了 LogSoftmax 和 NLLLoss，所以我们在模型的输出层不需要手动加 Softmax。

In [5]:
optimizer = optim.Adam(model.parameters(), lr=config.LEARNING_RATE)
loss_function = nn.CrossEntropyLoss()

print("Start Training...")
for epoch in range(config.EPOCHS):
    total_loss = 0
    for context, target in dataloader:
        # 搬运数据到 GPU (如果可用)
        context = context.to(device)
        target = target.to(device)
        
        # 1. 梯度清零
        model.zero_grad()
        
        # 2. 前向传播
        log_probs = model(context)
        
        # 3. 计算损失
        loss = loss_function(log_probs, target)
        
        # 4. 反向传播
        loss.backward()
        
        # 5. 更新参数
        optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch+1}/{config.EPOCHS}, Loss: {avg_loss:.4f}")

print("Training Finished!")

Start Training...
Epoch 1/10, Loss: 6.7122
Epoch 2/10, Loss: 5.6935
Epoch 3/10, Loss: 5.2916
Epoch 4/10, Loss: 5.0184
Epoch 5/10, Loss: 4.7951
Epoch 6/10, Loss: 4.6002
Epoch 7/10, Loss: 4.4319
Epoch 8/10, Loss: 4.2801
Epoch 9/10, Loss: 4.1454
Epoch 10/10, Loss: 4.0188
Training Finished!


## 第六阶段：模型应用与可视化
训练完了，怎么知道模型好不好？我们写一个函数，输入一个词，找出和它最相似的词。这通过计算余弦相似度 (Cosine Similarity) 来实现。

In [6]:
def get_similar_words(word, n=5):
    # 检查词是否在词表中
    if word not in vocab:
        print(f"Word '{word}' not in vocabulary.")
        return

    # 1. 取出该词的向量
    word_idx = vocab[word]
    # .cpu() 确保我们可以在 CPU 上做 numpy 运算
    word_vec = model.embeddings.weight[word_idx].cpu().detach().numpy()
    
    # 2. 取出所有词的向量矩阵
    all_weights = model.embeddings.weight.cpu().detach().numpy()
    
    # 3. 计算余弦相似度
    # Cosine Sim = (A . B) / (|A| * |B|)
    dot_product = np.dot(all_weights, word_vec)
    norm_all = np.linalg.norm(all_weights, axis=1)
    norm_word = np.linalg.norm(word_vec)
    
    similarities = dot_product / (norm_all * norm_word)
    
    # 4. 排序 (argsort 返回的是索引，[-n-1:-1] 取最大的 n 个，除了它自己)
    top_indices = np.argsort(similarities)[-n-1:-1][::-1]
    
    print(f"\nWords closest to '{word}':")
    for idx in top_indices:
        print(f"  {idx_to_word[idx]} (Sim: {similarities[idx]:.4f})")

# 测试几个《爱丽丝梦游仙境》里的典型词汇
get_similar_words("alice")
get_similar_words("queen")
get_similar_words("rabbit")
get_similar_words("tea")


Words closest to 'alice':
  exactly (Sim: 0.3279)
  rome (Sim: 0.3033)
  stop (Sim: 0.3032)
  done (Sim: 0.2981)
  listen (Sim: 0.2859)

Words closest to 'queen':
  sends (Sim: 0.3515)
  mouse (Sim: 0.3470)
  dish (Sim: 0.3455)
  only (Sim: 0.3246)
  pie (Sim: 0.3211)

Words closest to 'rabbit':
  felt (Sim: 0.3748)
  blow (Sim: 0.3481)
  executioner (Sim: 0.3355)
  fashion (Sim: 0.3210)
  staring (Sim: 0.3203)

Words closest to 'tea':
  will (Sim: 0.2942)
  dance (Sim: 0.2777)
  box (Sim: 0.2775)
  man (Sim: 0.2764)
  loudly (Sim: 0.2742)
